 # Bell Curve Grading Tool

In [ ]:
import polars as pl

@pl.api.register_expr_namespace("stats")
class Stats:
    def __init__(self, expr: pl.Expr):
        self._expr = expr
    
    def bell_curve(self, scores_column: str):
        scores_expr = pl.col(scores_column)
        
        mean_expr = scores_expr.mean().alias("mean")
        std_expr = scores_expr.std().alias("std")
        z_scores_expr = (scores_expr - mean_expr) / std_expr
        
        grading_expr = (
            pl.when(z_scores_expr >= 1.0).then(pl.lit("A"))
            .when((z_scores_expr >= 0.0) & (z_scores_expr < 1.0)).then(pl.lit("B"))
            .when((z_scores_expr >= -1.0) & (z_scores_expr < 0.0)).then(pl.lit("C"))
            .when((z_scores_expr >= -2.0) & (z_scores_expr < -1.0)).then(pl.lit("D"))
            .otherwise(pl.lit("F"))
            .alias("Grade")
        )

        return grading_expr

In [ ]:
df = pl.read_csv('../../datasets/student_scores.csv')
df

In [ ]:
df = df.with_columns(pl.col("Score").stats.bell_curve('Score'))
df